# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
mcp_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await mcp_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    mcp_client=mcp_client,
)

### reset monologue (optional)

In [2]:
ToolUseAgent.reset(project_name="gai",agent_name="ToolUseAgent")


### b) start_async

In [3]:
goal = "When is the next public holiday in Singapore?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

I'll help you find the next public holiday in Singapore. First, let me get the current date, and then search for Singapore's public holidays.Now let me search for Singapore's public holidays to find the next one after June 30, 2025.

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "292e4873-1f4f-41f3-b5cc-457a6868627c",
    "header": {
        "sender": "User",
        "recipient": "Assistant",
        "timestamp": 1751253365.4795945,
        "order": 0
    },
    "body": {
        "type": "state",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_0143ML6R72pQZiSzg9e9YHQM",
                "content": "Current UTC time is 2025-06-30, and the time in Asia/Singapore is 2025-06-30."
            }
        ]
    }
}
{
    "id": "0b354c6f-ea1b-4543-ad2a-ba90999c7cb0",
    "header": {
        "sender": "Assistant",
        "recipient": "User",
        "timestamp": 1751253368.6968157,
        "order": 0
    },
    "body": {
        "type": "state",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_

### d) continue_async

In [9]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


### e) interrupt_async

In [7]:
resp = await agent.interrupt_async(message="Tell me a one paragraph joke.")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


ERROR    async_anthropic_create: Error while calling anthropic API: Error code: 400 - {'type': 'error', 'error':   
         {'type': 'invalid_request_error', 'message': 'messages.10: `tool_use` ids were found without `tool_result`
         blocks immediately after: toolu_01JDYjoLAQZnKK1G4mS6Mw3Y. Each `tool_use` block must have a corresponding 
         `tool_result` block in the next message.'}}

WARNING  [tool_use] async_anthropic_create: Error while calling anthropic API: Error code: 400 - {'type': 'error', 
         'error': {'type': 'invalid_request_error', 'message': 'messages.10: `tool_use` ids were found without     
         `tool_result` blocks immediately after: toolu_01JDYjoLAQZnKK1G4mS6Mw3Y. Each `tool_use` block must have a 
         corresponding `tool_result` block in the next message.'}}. Retry in 5 seconds (1/5).

Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but sees nothing except normal bookshelves. He turns back to the librarian, who points to the psychology section and says, "No seriously, the paranoia books are literally right behind you on that shelf." The man laughs nervously and says, "Oh, I thought you meant..." The librarian interrupts, "That someone was following you? Sir, this is a library, not a spy movie. Though I have to say, your reaction was so dramatic, you might want to check out the acting section too – it's right behind the paranoia books."

Now, let me get back on track to help you find the next public holiday in Singapore. Since I'm having connectivity issues with external sources, let me try a different approach to get this information for you.I'm experiencing connectivity issues with the search and scraping tools. Based on my knowledge